In [11]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# -----------------------------
# LOAD DATA
# -----------------------------

df = pd.read_csv(
    "../../data/smart_water_dataset.csv"
)

# -----------------------------
# ENCODE SEASON
# -----------------------------

encoder = LabelEncoder()

df["season"] = encoder.fit_transform(
    df["season"]
)

# -----------------------------
# FEATURES
# -----------------------------

X = df[
    [
        "season",
        "rainfall_mm",
        "humidity",
        "temperature",
        "tank_capacity",
        "water_level",
        "daily_usage",
        "occupancy",
        "roof_area",
        "collection_efficiency"
    ]
]

# -----------------------------
# TARGET
# -----------------------------

y = df["water_collected"]

# -----------------------------
# TRAIN TEST SPLIT
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# -----------------------------
# MODELS
# -----------------------------

models = {

    "Linear Regression":
        LinearRegression(),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        ),

    "XGBoost":
        XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=6,
            random_state=42
        )
}

# -----------------------------
# TRAIN + EVALUATE
# -----------------------------

best_model = None
best_score = -999

for name, model in models.items():

    print(f"\nTRAINING: {name}")

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = mean_squared_error(
        y_test,
        predictions
    ) ** 0.5

    r2 = r2_score(
        y_test,
        predictions
    )

    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2 Score: {r2:.4f}")

    # SAVE BEST MODEL

    if r2 > best_score:
        best_score = r2
        best_model = model

# -----------------------------
# SAVE BEST MODEL
# -----------------------------

with open(
    "../../models/collection_model.pkl",
    "wb"
) as f:

    pickle.dump(best_model, f)

print("\nBEST MODEL SAVED!")

# -----------------------------
# SAVE ENCODER
# -----------------------------

with open(
    "../../models/season_encoder.pkl",
    "wb"
) as f:

    pickle.dump(encoder, f)

print("ENCODER SAVED!")


TRAINING: Linear Regression
MAE: 76.83
RMSE: 116.33
R2 Score: 0.8871

TRAINING: Random Forest
MAE: 5.88
RMSE: 11.28
R2 Score: 0.9989

TRAINING: XGBoost
MAE: 5.11
RMSE: 8.43
R2 Score: 0.9994

BEST MODEL SAVED!
ENCODER SAVED!
